# NB05 — Optimización, interoperabilidad y ONNX
**Correspondencia: Semanas 14–15**


## 1. Preparación del entorno


In [ ]:
!pip -q install tf2onnx onnx onnxruntime

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import onnx
import onnxruntime as ort

from tensorflow import keras
from pathlib import Path

print("TensorFlow:", tf.__version__)
print("ONNX Runtime:", ort.__version__)


## 2. Conectar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Recuperar el modelo definitivo

Utilice el modelo seleccionado al finalizar NB04.


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/MachineLearning2026/modelo_cnn_definitivo.keras"

model = keras.models.load_model(MODEL_PATH)

model.summary()


## 4. Definir las clases y el contrato del modelo

Mantenga exactamente el mismo orden de clases utilizado durante el entrenamiento.


In [ ]:
# Reemplace por las clases reales de su proyecto.
CLASS_NAMES = [
    "clase_1",
    "clase_2",
    "clase_3",
    "clase_4"
]

IMG_SIZE = (224, 224)

contrato = {
    "input_shape": [1, 224, 224, 3],
    "tipo_entrada": "float32",
    "canales": "RGB",
    "preprocesamiento": "MobileNetV2 preprocess_input",
    "clases": CLASS_NAMES
}

contrato


## 5. Preparar una imagen de prueba

Utilice una imagen que no haya sido utilizada durante el entrenamiento.


In [ ]:
from PIL import Image

IMAGE_PATH = "/content/drive/MyDrive/MachineLearning2026/imagen_prueba.jpg"

img = Image.open(IMAGE_PATH).convert("RGB")
img = img.resize(IMG_SIZE)

plt.figure(figsize=(5,5))
plt.imshow(img)
plt.axis("off")
plt.show()


## 6. Preprocesar la imagen


In [ ]:
x = np.array(img, dtype=np.float32)
x = np.expand_dims(x, axis=0)

x_preprocessed = keras.applications.mobilenet_v2.preprocess_input(x)

print("Shape:", x_preprocessed.shape)
print("Tipo:", x_preprocessed.dtype)


## 7. Baseline del modelo original

Antes de optimizar o convertir el modelo registraremos:

- tamaño;
- latencia de inferencia;
- predicción;
- confianza.

Estos valores constituyen la **baseline**.


In [ ]:
keras_size_mb = os.path.getsize(MODEL_PATH) / (1024 ** 2)

# Warm-up
for _ in range(5):
    _ = model.predict(x, verbose=0)

tiempos = []

for _ in range(30):
    inicio = time.perf_counter()
    pred = model.predict(x, verbose=0)
    fin = time.perf_counter()
    tiempos.append((fin - inicio) * 1000)

keras_latency = np.mean(tiempos)
keras_prediction = int(np.argmax(pred[0]))
keras_confidence = float(np.max(pred[0]))

print(f"Tamaño modelo Keras: {keras_size_mb:.2f} MB")
print(f"Latencia promedio: {keras_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[keras_prediction])
print(f"Confianza: {keras_confidence:.2%}")


## 8. Cuantificación

La cuantificación reduce la precisión numérica utilizada para representar determinados valores del modelo.

En esta práctica generaremos una versión **TensorFlow Lite cuantificada dinámicamente** para observar el efecto sobre el tamaño del archivo.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

TFLITE_PATH = "/content/modelo_quantized.tflite"

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_quant_model)

tflite_size_mb = os.path.getsize(TFLITE_PATH) / (1024 ** 2)

print(f"Tamaño TFLite cuantificado: {tflite_size_mb:.2f} MB")


## 9. Inferencia con el modelo cuantificado


In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Entrada:", input_details[0]["shape"], input_details[0]["dtype"])
print("Salida:", output_details[0]["shape"], output_details[0]["dtype"])


In [ ]:
tflite_input = x.astype(input_details[0]["dtype"])

# Warm-up
for _ in range(5):
    interpreter.set_tensor(input_details[0]["index"], tflite_input)
    interpreter.invoke()

tiempos_tflite = []

for _ in range(30):
    interpreter.set_tensor(input_details[0]["index"], tflite_input)

    inicio = time.perf_counter()
    interpreter.invoke()
    fin = time.perf_counter()

    tiempos_tflite.append((fin - inicio) * 1000)

tflite_pred = interpreter.get_tensor(output_details[0]["index"])

tflite_latency = np.mean(tiempos_tflite)
tflite_prediction = int(np.argmax(tflite_pred[0]))
tflite_confidence = float(np.max(tflite_pred[0]))

print(f"Latencia TFLite: {tflite_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[tflite_prediction])
print(f"Confianza: {tflite_confidence:.2%}")


## 10. Comparar modelo original y cuantificado


In [ ]:
comparacion_optimizacion = pd.DataFrame({
    "Modelo": ["Keras original", "TFLite cuantificado"],
    "Tamaño_MB": [keras_size_mb, tflite_size_mb],
    "Latencia_ms": [keras_latency, tflite_latency],
    "Clase_predicha": [
        CLASS_NAMES[keras_prediction],
        CLASS_NAMES[tflite_prediction]
    ],
    "Confianza": [
        keras_confidence,
        tflite_confidence
    ]
})

comparacion_optimizacion


## 11. Poda de redes

La poda (*pruning*) busca reducir conexiones o pesos de baja relevancia.

En esta práctica la abordaremos mediante una demostración sencilla de **poda por magnitud** sobre los pesos del modelo. El objetivo es observar el concepto de **sparsity** sin reemplazar todavía el modelo definitivo.


In [ ]:
pesos = model.get_weights()

total = 0
ceros_originales = 0

for w in pesos:
    total += w.size
    ceros_originales += np.count_nonzero(w == 0)

sparsity_original = ceros_originales / total

print(f"Sparsity original: {sparsity_original:.2%}")


## 12. Simular poda por magnitud


In [ ]:
pesos_podados = []
valores_absolutos = np.concatenate([
    np.abs(w.flatten())
    for w in pesos
    if w.ndim > 1
])

umbral = np.percentile(valores_absolutos, 30)

for w in pesos:
    w_nuevo = w.copy()

    if w.ndim > 1:
        w_nuevo[np.abs(w_nuevo) < umbral] = 0

    pesos_podados.append(w_nuevo)

total_podado = sum(w.size for w in pesos_podados)
ceros_podados = sum(np.count_nonzero(w == 0) for w in pesos_podados)

sparsity_podada = ceros_podados / total_podado

print(f"Umbral aplicado: {umbral:.6f}")
print(f"Sparsity después de la poda: {sparsity_podada:.2%}")


## 13. Crear una copia del modelo con pesos podados

Esta versión se utiliza solamente para comparar el efecto de la poda.


In [ ]:
pruned_model = keras.models.clone_model(model)
pruned_model.set_weights(pesos_podados)

pred_pruned = pruned_model.predict(x, verbose=0)

pruned_prediction = int(np.argmax(pred_pruned[0]))
pruned_confidence = float(np.max(pred_pruned[0]))

print("Clase original:", CLASS_NAMES[keras_prediction])
print("Clase después de poda:", CLASS_NAMES[pruned_prediction])
print(f"Confianza original: {keras_confidence:.2%}")
print(f"Confianza después de poda: {pruned_confidence:.2%}")


## 14. Interpretar la poda

La poda no debe evaluarse solamente sobre una imagen. En el proyecto deberá analizarse utilizando un conjunto de datos y métricas de desempeño.

Observe además que aumentar la cantidad de ceros **no garantiza automáticamente** una reducción equivalente del tamaño del archivo ni una mejora de latencia. El beneficio depende del formato y del runtime utilizado.


## 15. Exportar el modelo a ONNX

ONNX permite representar el modelo en un formato interoperable para ejecutarlo posteriormente mediante runtimes diferentes del framework original.


In [ ]:
import tf2onnx

ONNX_PATH = "/content/modelo_cnn.onnx"

input_signature = (
    tf.TensorSpec(
        (None, IMG_SIZE[0], IMG_SIZE[1], 3),
        tf.float32,
        name="input"
    ),
)

onnx_model, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=13
)

with open(ONNX_PATH, "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Modelo ONNX generado:")
print(ONNX_PATH)


## 16. Validar el archivo ONNX


In [ ]:
onnx_model_check = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model_check)

onnx_size_mb = os.path.getsize(ONNX_PATH) / (1024 ** 2)

print("Modelo ONNX válido")
print(f"Tamaño: {onnx_size_mb:.2f} MB")


## 17. Cargar ONNX con ONNX Runtime


In [ ]:
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("Input:", input_name)
print("Input shape:", session.get_inputs()[0].shape)
print("Output:", output_name)
print("Output shape:", session.get_outputs()[0].shape)


## 18. Inferencia mediante ONNX Runtime

El modelo exportado debe recibir exactamente el mismo tipo de entrada que el modelo original.


In [ ]:
onnx_input = x.astype(np.float32)

# Warm-up
for _ in range(5):
    _ = session.run(
        [output_name],
        {input_name: onnx_input}
    )

tiempos_onnx = []

for _ in range(30):
    inicio = time.perf_counter()

    onnx_pred = session.run(
        [output_name],
        {input_name: onnx_input}
    )[0]

    fin = time.perf_counter()
    tiempos_onnx.append((fin - inicio) * 1000)

onnx_latency = np.mean(tiempos_onnx)
onnx_prediction = int(np.argmax(onnx_pred[0]))
onnx_confidence = float(np.max(onnx_pred[0]))

print(f"Latencia ONNX Runtime: {onnx_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[onnx_prediction])
print(f"Confianza: {onnx_confidence:.2%}")


## 19. Comparar Keras y ONNX


In [ ]:
comparacion_onnx = pd.DataFrame({
    "Modelo": ["Keras", "ONNX"],
    "Tamaño_MB": [keras_size_mb, onnx_size_mb],
    "Latencia_ms": [keras_latency, onnx_latency],
    "Clase_predicha": [
        CLASS_NAMES[keras_prediction],
        CLASS_NAMES[onnx_prediction]
    ],
    "Confianza": [
        keras_confidence,
        onnx_confidence
    ]
})

comparacion_onnx


## 20. Verificar equivalencia numérica


In [ ]:
diferencia_maxima = np.max(
    np.abs(pred[0] - onnx_pred[0])
)

print(f"Diferencia máxima Keras vs ONNX: {diferencia_maxima:.8f}")


## 21. Guardar los artefactos para la implementación

Los archivos generados se almacenarán en Google Drive para utilizarlos posteriormente en NB06.


In [ ]:
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/MachineLearning2026/implementacion"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import shutil

shutil.copy(ONNX_PATH, OUTPUT_DIR / "modelo_cnn.onnx")
shutil.copy(TFLITE_PATH, OUTPUT_DIR / "modelo_quantized.tflite")

print("Archivos guardados en:")
print(OUTPUT_DIR)


## 22. Guardar el orden de clases


In [ ]:
import json

CLASSES_PATH = OUTPUT_DIR / "class_names.json"

with open(CLASSES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        CLASS_NAMES,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Clases guardadas en:")
print(CLASSES_PATH)


## 23. Comparación final

Complete la tabla con las métricas obtenidas para las versiones que correspondan a su proyecto.


In [ ]:
comparacion_final = pd.DataFrame({
    "Versión": [
        "Keras original",
        "TFLite cuantificado",
        "ONNX"
    ],
    "Tamaño_MB": [
        keras_size_mb,
        tflite_size_mb,
        onnx_size_mb
    ],
    "Latencia_ms": [
        keras_latency,
        tflite_latency,
        onnx_latency
    ],
    "Confianza_imagen_prueba": [
        keras_confidence,
        tflite_confidence,
        onnx_confidence
    ]
})

comparacion_final


## 24. Actividad

A partir del modelo definitivo del proyecto integrador:

1. Registre tamaño y latencia del modelo original.
2. Defina una baseline antes de optimizar.
3. Genere una versión cuantificada.
4. Compare tamaño, latencia y predicción con el modelo original.
5. Analice conceptualmente la poda y mida la sparsity obtenida.
6. Determine qué efectos podría tener una poda excesiva.
7. Exporte el modelo a ONNX.
8. Valide la estructura del archivo ONNX.
9. Ejecute inferencia mediante ONNX Runtime.
10. Compare la predicción de Keras y ONNX.
11. Compruebe que el contrato de entrada y salida se conserva.
12. Justifique qué versión utilizaría para la implementación del producto final.


## 25. Base para la Evaluación 4

Este notebook inicia la etapa técnica de la **Evaluación 4**.

El resultado esperado es disponer de evidencia sobre eficiencia e interoperabilidad y de un modelo preparado para integrarse posteriormente con una aplicación.

**Modelo definitivo → baseline → cuantificación/poda → comparación → ONNX → ONNX Runtime → modelo preparado para implementación**

En NB06 se utilizarán estos artefactos para construir la aplicación con **Streamlit**, organizar el proyecto en **GitHub** y realizar el despliegue mediante **Streamlit Community Cloud**.
